# 01. EDA: REES46 multi-category store (Pair 1, week 2)

Goal: understand the interaction data before ETL and ALS.

Questions this notebook answers:
1. How many users, products and events; how sparse is the user-item matrix?
2. Event type mix and the view -> cart -> purchase funnel.
3. Daily activity over the month.
4. Top categories and brands; missing `category_code` / `brand`.
5. Price distribution.
6. Interactions per user and per product (long tail), user segments (new / light / heavy) for week 7.
7. Thresholds to filter rare users/products before ALS.

**Input:** the development sample written by `ingestion/inspect_raw.py` (default) or a full month CSV.
Set `DATA_PATH` below. Run Jupyter from the repository root so `common/` is importable.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "common").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F

from common.schemas import RAW_COLUMNS, CLEAN_PARTITION_COLUMN, spark_event_schema
from common.weights import EVENT_WEIGHTS

# Sample: python -m ingestion.inspect_raw data/raw/2019-Oct.csv --limit 2000000 \
#             --sample-out data/sample/oct_sample.csv --sample-rows 200000
# Full month: EDA_DATA=data/raw/2019-Oct.csv (relative to the repository root)
DATA_PATH = ROOT / os.getenv("EDA_DATA", "data/sample/oct_sample.csv")

In [ ]:
spark = (
    SparkSession.builder
    .appName("eda-rees46")
    .master("local[*]")
    .config("spark.driver.memory", os.getenv("EDA_DRIVER_MEMORY", "6g"))
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

raw = spark.read.csv(str(DATA_PATH), header=True, schema=spark_event_schema())
assert tuple(raw.columns) == RAW_COLUMNS, raw.columns

# event_time is "2019-10-01 00:00:00 UTC" in the raw files
df = (
    raw.withColumn("event_time", F.to_timestamp(F.regexp_replace("event_time", " UTC$", ""), "yyyy-MM-dd HH:mm:ss"))
       .withColumn(CLEAN_PARTITION_COLUMN, F.to_date("event_time"))
       .cache()
)
df.printSchema()
df.show(5, truncate=False)

## 1. Size and sparsity

In [ ]:
n_events = df.count()
n_users = df.select("user_id").distinct().count()
n_items = df.select("product_id").distinct().count()
n_pairs = df.select("user_id", "product_id").distinct().count()
sparsity = 1 - n_pairs / (n_users * n_items)

print(f"events        {n_events:,}")
print(f"users         {n_users:,}")
print(f"products      {n_items:,}")
print(f"user-item pairs {n_pairs:,}")
print(f"sparsity      {sparsity:.6%}")
# format in Spark: collecting a timestamp converts it to the local time zone of the machine
fmt = "yyyy-MM-dd HH:mm:ss"
print("time range (UTC)", *df.agg(F.date_format(F.min("event_time"), fmt), F.date_format(F.max("event_time"), fmt)).first())

## 2. Event types and funnel

In [ ]:
by_type = df.groupBy("event_type").count().orderBy(F.desc("count")).toPandas()
by_type["share"] = by_type["count"] / by_type["count"].sum()
by_type["weight"] = by_type["event_type"].map(EVENT_WEIGHTS)
by_type

In [ ]:
# Users who reached each step at least once
funnel = (
    df.groupBy("user_id")
      .agg(*[F.max(F.when(F.col("event_type") == t, 1).otherwise(0)).alias(t) for t in ("view", "cart", "purchase")])
      .agg(*[F.sum(t).alias(t) for t in ("view", "cart", "purchase")])
      .toPandas()
)
funnel

## 3. Daily activity

In [ ]:
daily = (
    df.groupBy(CLEAN_PARTITION_COLUMN, "event_type").count()
      .groupBy(CLEAN_PARTITION_COLUMN).pivot("event_type").sum("count")
      .orderBy(CLEAN_PARTITION_COLUMN).toPandas().set_index(CLEAN_PARTITION_COLUMN)
)
daily.plot(subplots=True, figsize=(10, 6), title="Events per day")
plt.tight_layout()

## 4. Categories and brands

In [ ]:
missing = df.select(
    F.mean(F.col("category_code").isNull().cast("int")).alias("category_code_missing"),
    F.mean(F.col("brand").isNull().cast("int")).alias("brand_missing"),
).toPandas()
missing

In [ ]:
top_categories = (
    df.withColumn("top_level", F.split("category_code", r"\.").getItem(0))
      .groupBy("top_level").agg(F.count("*").alias("events"), F.countDistinct("product_id").alias("products"))
      .orderBy(F.desc("events")).limit(20).toPandas()
)
top_brands = (
    df.where(F.col("brand").isNotNull())
      .groupBy("brand").agg(F.count("*").alias("events"), F.sum((F.col("event_type") == "purchase").cast("int")).alias("purchases"))
      .orderBy(F.desc("events")).limit(20).toPandas()
)
display(top_categories, top_brands)

## 5. Price

In [ ]:
df.select("price").summary("count", "min", "25%", "50%", "75%", "max", "mean").show()
print("zero or negative prices:", df.where(F.col("price") <= 0).count())

price_pd = df.where(F.col("price") > 0).select("price").sample(fraction=min(1.0, 200_000 / n_events), seed=42).toPandas()
price_pd["price"].plot.hist(bins=100, logy=True, figsize=(8, 3), title="Price (log count)")

## 6. Interactions per user and per product

Interaction score per (user, product) = sum of event weights (`common/weights.py`), the same value the ALS confidence uses.

In [ ]:
weights_expr = F.create_map(*[x for k, v in EVENT_WEIGHTS.items() for x in (F.lit(k), F.lit(v))])
pairs = (
    df.withColumn("w", weights_expr[F.col("event_type")])
      .groupBy("user_id", "product_id").agg(F.sum("w").alias("score"), F.count("*").alias("events"))
      .cache()
)

per_user = pairs.groupBy("user_id").agg(F.count("*").alias("n_items"), F.sum("events").alias("n_events"))
per_item = pairs.groupBy("product_id").agg(F.count("*").alias("n_users"))

for name, frame, col in (("per user", per_user, "n_items"), ("per product", per_item, "n_users")):
    print(name)
    frame.select(col).summary("min", "25%", "50%", "75%", "90%", "99%", "max").show()

In [ ]:
# Segments for week 7 (thresholds to be confirmed on the full train split)
segments = (
    per_user.withColumn(
        "segment",
        F.when(F.col("n_items") <= 2, "new (<=2 items)")
         .when(F.col("n_items") <= 10, "light (3-10)")
         .otherwise("heavy (>10)"),
    )
    .groupBy("segment").agg(F.count("*").alias("users"), F.sum("n_events").alias("events"))
    .toPandas()
)
segments["user_share"] = segments["users"] / segments["users"].sum()
segments["event_share"] = segments["events"] / segments["events"].sum()
segments

In [ ]:
# Long tail: share of interactions covered by the top x% of products
item_pd = per_item.orderBy(F.desc("n_users")).toPandas()
cum = item_pd["n_users"].cumsum() / item_pd["n_users"].sum()
plt.figure(figsize=(6, 3))
plt.plot((item_pd.index + 1) / len(item_pd), cum)
plt.xlabel("share of products"); plt.ylabel("share of user-item pairs"); plt.title("Product popularity long tail")

## 7. Filtering thresholds for ALS

In [ ]:
# How much data survives a minimum number of users per product / products per user
rows = []
for min_users in (1, 2, 5, 10):
    for min_items in (1, 2, 5):
        keep_items = per_item.where(F.col("n_users") >= min_users).select("product_id")
        keep_users = per_user.where(F.col("n_items") >= min_items).select("user_id")
        kept = pairs.join(keep_items, "product_id").join(keep_users, "user_id")
        rows.append((min_users, min_items, kept.count(), kept.select("user_id").distinct().count(),
                     kept.select("product_id").distinct().count()))
import pandas as pd
pd.DataFrame(rows, columns=["min_users_per_item", "min_items_per_user", "pairs", "users", "products"])

## 8. Findings (full October 2019, 42,448,764 events)

- **Sparsity:** 3,022,290 users x 166,794 products, 23,307,630 distinct user-product pairs, so 99.9954% of the matrix is empty.
- **Event mix / funnel:** view 96.07%, cart 2.18%, purchase 1.75%. 3,022,130 users viewed, 337,117 added to cart, 347,118 purchased: more users purchase than add to cart, so ETL and scoring must not assume a cart event before a purchase.
- **Daily activity:** 1.13M to 1.64M events per day (min Oct 3, max Oct 13, from `docs/data-check-oct.json`); no gaps.
- **Missing values to handle in ETL:** `category_code` 31.8% and `brand` 14.4% empty; keep them as null (`category_id` is always present, so `pop:cat` uses `category_id`). `user_session` is empty in 2 rows.
- **Price anomalies:** 68,673 events (0.16%) have price 0; median 162.93, max 2,574.07. Price is not used by ALS, so keep these events for interactions and exclude them only from price statistics and the UI.
- **Long tail:** users per product: median 17, p90 198, p99 1,946, max 197,878. Products per user: median 3, p90 19, p99 69.
- **Segment thresholds chosen:** new <= 2 products (46.6% of users, 7.1% of events), light 3-10 (34.2%, 22.8%), heavy > 10 (19.2%, 70.2%). To be re-checked on the train split in week 7.
- **ALS filter (proposal):** at least 5 users per product and 2 products per user. Keeps 127,875 products (76.7%), 2,029,473 users (67.2%) and 95.4% of the user-product pairs. Users with a single product are served by the popularity / trending fallback.
- **Notes for the time-based split (train Oct 1 - Nov 15, test Nov 16 - Nov 30):** this notebook covers October only; `2019-Nov.csv` (9.0 GB) is needed for the train/test split, and the share of test users never seen in train must be measured after ETL.
